In [3]:
import re
from urllib.parse import urlparse, parse_qs
from dotenv import load_dotenv
from youtube_transcript_api import YouTubeTranscriptApi
from youtube_transcript_api._errors import TranscriptsDisabled, NoTranscriptFound
from langchain_huggingface import HuggingFaceEndpointEmbeddings, ChatHuggingFace, HuggingFaceEndpoint
from langchain_community.vectorstores import FAISS
from langchain_core.documents import Document
from langchain_core.messages import HumanMessage
from langchain_core.prompts import PromptTemplate

load_dotenv()

True

In [4]:

_PLAIN_ID_RE = re.compile(r"^[A-Za-z0-9_-]{11}$")

def extract_video_id(url_or_id: str) -> str:
    """
    Parse any YouTube URL or plain 11-char ID and return the video ID.

    Supports:
      https://www.youtube.com/watch?v=dQw4w9WgXcQ
      https://youtu.be/dQw4w9WgXcQ
      https://youtube.com/shorts/dQw4w9WgXcQ
      dQw4w9WgXcQ  (plain ID)
    """
    value = url_or_id.strip()

    if _PLAIN_ID_RE.match(value):
        return value

    parsed = urlparse(value)
    host = parsed.netloc.lower().removeprefix("www.")

    if host in ("youtube.com", "m.youtube.com"):
        qs = parse_qs(parsed.query)
        if "v" in qs and qs["v"]:
            vid = qs["v"][0]
            if _PLAIN_ID_RE.match(vid):
                return vid
        parts = [p for p in parsed.path.split("/") if p]
        if len(parts) >= 2 and parts[0] in ("shorts", "embed", "v", "e"):
            vid = parts[1]
            if _PLAIN_ID_RE.match(vid):
                return vid

    elif host == "youtu.be":
        vid = parsed.path.lstrip("/").split("?")[0]
        if _PLAIN_ID_RE.match(vid):
            return vid

    raise ValueError(f"Could not extract a YouTube video ID from: '{value}'")


def format_timestamp(seconds: float) -> str:
    """Convert raw seconds → MM:SS string."""
    total = int(seconds)
    return f"{total // 60:02d}:{total % 60:02d}"


def youtube_link(video_id: str, seconds: float) -> str:
    """Return a YouTube deep-link that starts at the given second."""
    return f"https://youtu.be/{video_id}?t={int(seconds)}"



print( "dQw4w9WgXcQ: ",extract_video_id("dQw4w9WgXcQ"))
print("https://youtu.be/dQw4w9WgXcQ: ",extract_video_id("https://youtu.be/dQw4w9WgXcQ"))
print("https://www.youtube.com/watch?v=dQw4w9WgXcQ: ",extract_video_id("https://www.youtube.com/watch?v=dQw4w9WgXcQ"))
print("formated time: ",format_timestamp(180))
print("youtube_link with timestamp: ",youtube_link("a3C1DMswClQ",12))

dQw4w9WgXcQ:  dQw4w9WgXcQ
https://youtu.be/dQw4w9WgXcQ:  dQw4w9WgXcQ
https://www.youtube.com/watch?v=dQw4w9WgXcQ:  dQw4w9WgXcQ
formated time:  03:00
youtube_link with timestamp:  https://youtu.be/a3C1DMswClQ?t=12


In [5]:

PREFERRED_LANGS = ["en", "en-US", "en-GB", "hi"]

def fetch_transcript(video_id: str) -> list:
    """
    Fetch the best available transcript for a video.

    Returns a list of transcript entries where each entry has:
      .text      - spoken text
      .start     - offset in seconds from start of video
      .duration  - duration of the segment in seconds
    """
    api = YouTubeTranscriptApi()

    try:
        transcript_list = api.list(video_id)
    except TranscriptsDisabled:
        raise RuntimeError(f"Transcripts are disabled for video '{video_id}'.")
    except Exception as e:
        raise RuntimeError(f"Could not fetch transcript for '{video_id}': {e}")

    print("Available transcripts:")
    for t in transcript_list:
        print(f"  {t.language_code} – {t.language}")

    try:
        transcript = transcript_list.find_transcript(PREFERRED_LANGS).fetch()
        print("Preferred language transcript fetched.")
    except NoTranscriptFound:
        transcript = next(iter(transcript_list)).fetch()
        print("Preferred language not found — using fallback.")

    return transcript


def full_text_from_transcript(transcript: list) -> str:
    """Join all transcript segments into one string (used for summarisation)."""
    return " ".join(entry.text for entry in transcript)


In [7]:
    
data = fetch_transcript("a3C1DMswClQ")

print("fetched trranscript: ",full_text_from_transcript(data)[:500])

Available transcripts:
  en – English (auto-generated)
Preferred language transcript fetched.
fetched trranscript:  backend is huge and if we start discussing every single component that could be part of it we will be stuck here for years so what we will do is we will only discuss those topics which are used in majority of the code bases let's say 90% of them with that in mind let's talk about HTTP protocol the medium through which our browsers talk to our servers either to send data or to receive data from it and as I said there are a lot of other ways and protocols which clients and servers use to communica
